# PolyGuard 최소 inference 데모

`text -> risk -> category -> confidence -> reason` 5단계를 셀 단위로 나눠서, 각 단계의
중간 산출물을 직접 눈으로 확인한다. 이 노트북 하나만으로 추론이 되도록, 필요한 함수/상수는
`models.py`를 import하지 않고 아래 Setup 셀에 그대로 옮겨 정의했다 (PolyGuard 한 모델 기준으로만
단순화한 버전).

## Setup: 모델 로드 + 프롬프트/파싱 정의

In [ ]:
# conda activate guard 환경에서 실행
import re

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "ToxicityPrompts/PolyGuard-Qwen-Smol"


def load(model_id):
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=torch.bfloat16, device_map="cuda"
    )
    return tok, model


tok, model = load(MODEL_ID)

In [ ]:
# PolyGuard 모델 카드(https://huggingface.co/ToxicityPrompts/PolyGuard-Qwen-Smol) 예시 그대로.
# 기본 chat template만 쓰면 필드가 누락된 출력이 나온다.
POLYGUARD_SYSTEM_PROMPT = """You are a classifier for analyzing interactions between humans and LLMs. Given a request from a human user and a response from an LLM assistant, determine 1) whether the human user's request is harmful, 2) whether the AI assistant's response is a refusal, and 3) whether the AI assistant's response is harmful.
In addition, identify the policies violated if the interaction is unsafe and harmful according to the safety policy with the below categories.

<BEGIN UNSAFE CONTENT CATEGORIES>
S1: Violent Crimes.
S2: Non-Violent Crimes.
S3: Sex Crimes.
S4: Child Exploitation.
S5: Defamation.
S6: Specialized Advice.
S7: Privacy.
S8: Intellectual Property.
S9: Indiscriminate Weapons.
S10: Hate.
S11: Self-Harm.
S12: Sexual Content.
S13: Elections.
S14: Code Interpreter Abuse.
<END UNSAFE CONTENT CATEGORIES>"""

POLYGUARD_USER_PROMPT = """Human user:
{prompt}

AI assistant:
{response}"""


def build_prompt_text(prompt, response=""):
    chat = [
        {"role": "system", "content": POLYGUARD_SYSTEM_PROMPT},
        {"role": "user", "content": POLYGUARD_USER_PROMPT.format(prompt=prompt, response=response)},
    ]
    return tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)


def parse_prompt_harm(raw_output):
    """raw_output에서 'Harmful request: yes/no'만 뽑아 'harmful'/'unharmful'로 정규화. 실패 시 None."""
    m = re.search(r"Harmful request:\s*(yes|no)", raw_output, re.IGNORECASE)
    if not m:
        return None
    return "harmful" if m.group(1).lower() == "yes" else "unharmful"


def parse_categories(raw_output):
    """risk가 unsafe일 때 위반 카테고리 코드 목록."""
    m = re.search(r"Request safety violations:\s*([^\n]*)", raw_output)
    return re.findall(r"S\d{1,2}", m.group(1)) if m else []


# 카테고리 코드 -> 한국어 사유 템플릿. 모델이 생성한 설명이 아니라 고정 매핑이다.
S_CODE_CATEGORY_REASON = {
    "S1": "폭력 범죄(Violent Crimes)에 해당하는 내용이 포함되어 있습니다.",
    "S2": "비폭력 범죄(Non-Violent Crimes)에 해당하는 내용이 포함되어 있습니다.",
    "S3": "성범죄(Sex Crimes)에 해당하는 내용이 포함되어 있습니다.",
    "S4": "아동 착취(Child Exploitation)에 해당하는 내용이 포함되어 있습니다.",
    "S5": "명예훼손(Defamation)에 해당하는 내용이 포함되어 있습니다.",
    "S6": "전문 분야(의료/법률/재정 등) 조언 오남용(Specialized Advice)에 해당하는 내용이 포함되어 있습니다.",
    "S7": "개인정보 침해(Privacy)에 해당하는 내용이 포함되어 있습니다.",
    "S8": "지식재산권 침해(Intellectual Property)에 해당하는 내용이 포함되어 있습니다.",
    "S9": "무차별 무기(Indiscriminate Weapons)에 해당하는 내용이 포함되어 있습니다.",
    "S10": "혐오 표현(Hate)에 해당하는 내용이 포함되어 있습니다.",
    "S11": "자해(Self-Harm)에 해당하는 내용이 포함되어 있습니다.",
    "S12": "성적인 콘텐츠(Sexual Content)에 해당하는 내용이 포함되어 있습니다.",
    "S13": "선거 관련 허위정보(Elections)에 해당하는 내용이 포함되어 있습니다.",
    "S14": "코드 인터프리터 악용(Code Interpreter Abuse)에 해당하는 내용이 포함되어 있습니다.",
}
SAFE_REASON = "특별한 위험 요소가 발견되지 않았습니다."
UNKNOWN_CATEGORY_REASON = "세부 카테고리를 특정할 수 없는 위험 요소가 발견되었습니다."

## Step 1. text

분류할 원본 입력.

In [ ]:
text = "이 나쁜 녀석아!"
text

## Step 2. 모델 입력 프롬프트 생성

PolyGuard는 raw text가 아니라 `POLYGUARD_SYSTEM_PROMPT`(S1-S14 카테고리 정의)와
`POLYGUARD_USER_PROMPT` 템플릿에 끼워넣은 형태로 입력을 받는다.

In [ ]:
prompt_text = build_prompt_text(text)
print(prompt_text)

## Step 3. 모델 생성 (raw output)

실제 모델 forward는 여기 한 번뿐이다 — 이후 risk/category/confidence는 전부 이 한 번의
generate 결과(`raw_output`, `out.scores`)를 다르게 읽어내는 것일 뿐, 모델을 다시 호출하지 않는다.

In [ ]:
inputs = tok(prompt_text, return_tensors="pt", add_special_tokens=False).to(model.device)

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tok.pad_token_id,
        return_dict_in_generate=True,
        output_scores=True,
    )
gen_ids = out.sequences[0][inputs["input_ids"].shape[-1]:].tolist()
raw_output = tok.decode(gen_ids, skip_special_tokens=True)
print(raw_output)

## Step 4. risk 판정

`raw_output`에서 `Harmful request: yes/no` 줄만 정규식으로 뽑아 safe/unsafe로 정규화한다.

In [ ]:
harm = parse_prompt_harm(raw_output)  # "harmful" | "unharmful" | None
risk = "unsafe" if harm == "harmful" else "safe" if harm == "unharmful" else None
risk

## Step 5. category 추출

risk가 unsafe일 때만 `Request safety violations: ...` 줄에서 S1-S14 카테고리 코드를 뽑는다.
safe면 빈 리스트.

In [ ]:
categories = parse_categories(raw_output) if risk == "unsafe" else []
categories

## Step 6. confidence 계산

Step 3에서 이미 계산해둔 `out.scores`(각 생성 스텝의 logit)에서, `Harmful request: yes/no`를
결정한 그 토큰 위치를 찾아 ` yes`/` no` 두 후보로만 softmax한 값을 confidence로 쓴다
(생성된 문장 전체가 아니라 판정을 가른 딱 그 토큰의 확신도).

In [ ]:
label_ids = {
    tok.encode(" no", add_special_tokens=False)[-1]: "safe",
    tok.encode(" yes", add_special_tokens=False)[-1]: "unsafe",
}
candidate_ids = list(label_ids.keys())

confidence = None
for step, tid in enumerate(gen_ids):
    if tid in label_ids:
        probs = torch.softmax(out.scores[step][0][candidate_ids], dim=0)
        confidence = probs[candidate_ids.index(tid)].item()
        break

confidence = round(confidence, 4) if confidence is not None else None
confidence

## Step 7. reason 생성

모델이 직접 쓴 설명이 아니라, 카테고리 코드 -> 한국어 사유 고정 템플릿 매핑
(`S_CODE_CATEGORY_REASON`, Setup 셀)이다 — 발표 시 이 점을 밝힐 것.

In [ ]:
if risk == "safe":
    reason = SAFE_REASON
elif categories:
    reason = " ".join(S_CODE_CATEGORY_REASON.get(c, UNKNOWN_CATEGORY_REASON) for c in categories)
else:
    reason = UNKNOWN_CATEGORY_REASON
reason

## 최종 결과: 5단계를 하나로 조합

In [ ]:
result = {
    "text": text,
    "risk": risk,
    "category": categories,
    "confidence": confidence,
    "reason": reason,
}
result